# TP1: Introduction to PySpark (DataFrame APIs)

**Level:** Introductory

**Objectives**:
- Understand SparkSession and DataFrame basics.
- Read and write CSV/Parquet data.
- Practice common transformations, joins and aggregations.
- Learn caching/persistence and how to read execution plans.

You will work with small synthetic datasets and produce a short report (CSV + plot).

## Prerequisites
- PySpark available on the Jupyter kernel (Spark 3.3+ recommended)
- Python packages: `pandas`, `matplotlib`, `pyarrow` (for Parquet)

When running on JupyterHub, run the cells in order. If Spark is not configured, ask the system administrator to set `PYSPARK_SUBMIT_ARGS` or install pyspark on the kernel.

## Quick help
### Quick Spark / DataFrame reminder (cheat sheet)
- Create SparkSession: `from pyspark.sql import SparkSession; spark = SparkSession.builder.appName("app").getOrCreate()`
- Read CSV: `spark.read.option("header",True).csv("path")`
- Read Parquet: `spark.read.parquet("path")`
- Select columns: `df.select("col1","col2")`
- Filter rows: `df.filter(df.col > 10)` or `df.where("col > 10")`
- Add/modify column: `df.withColumn("new", expr(...))` or `df.withColumn("new", df.col * 2)`
- Cast column: `df.withColumn("ts", col("ts").cast("timestamp"))`
- Join: `df1.join(df2, on="key", how="left")`
- Aggregations: `df.groupBy("key").agg(count("*").alias("n"), sum("amount").alias("total"))`
- Cache/Persist: `df.cache()` then trigger with an action like `df.count()`
- Explain plan: `df.explain(True)` to see logical/physical plans
- Convert to pandas (small results): `df.toPandas()`
- Write Parquet: `df.write.mode("overwrite").parquet("out/")`
- For streaming: `spark.readStream.format("kafka")...` and `df.writeStream...start()`
- Use `checkpointLocation` for streaming durability


In [ ]:
# Cell 1: Setup - prepare workspace and imports
import os, shutil, sys
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
python_path = sys.executable

print("Driver sys.executable:", sys.executable)
print("Driver sys.version:", sys.version)

# Force workers to use same Python as driver
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Reset folders for a clean run
for d in ['data','output']:
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)

print('Workspace initialized')

In [ ]:
# Cell 2: Start SparkSession
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .master("local[3]")
         .appName('TP1_Introduction_PySpark')
         .config('spark.sql.shuffle.partitions','4')
         .config("spark.pyspark.python", python_path)                   # workers
         .config("spark.pyspark.driver.python", python_path)            # driver
         .config("spark.executorEnv.PYSPARK_PYTHON", python_path)       # ensure executors get this env
         .config("spark.yarn.appMasterEnv.PYSPARK_PYTHON", python_path) # for YARN appmaster
         .getOrCreate())
print('Spark version:', spark.version)

In [ ]:
# Check Spark execution environment
# Environment check: master, parallelism, executors
sc = spark.sparkContext
print("master:", sc.master)
print("defaultParallelism:", sc.defaultParallelism)

In [ ]:
# Cell 3: Create example datasets (CSV)
users = pd.DataFrame({
    'user_id':[1,2,3,4],
    'name':['Alice','Bob','Charlie','Diana'],
    'country':['FR','FR','DE','UK'],
    'signup_date':pd.to_datetime(['2023-01-10','2023-03-05','2022-11-20','2023-06-01'])
})
users.to_csv('data/users.csv', index=False)

# transactions: realistic-ish random events
rng = np.random.default_rng(42)
n = 1000
start = datetime(2024,1,1)
timestamps = [start + timedelta(minutes=int(x)) for x in rng.integers(0, 60*24*30, size=n)]
tx = pd.DataFrame({
    'tx_id': range(1,n+1),
    'user_id': rng.integers(1,5,size=n),
    'amount': np.round(rng.uniform(1.0,200.0,size=n),2),
    'currency': rng.choice(['EUR','GBP','USD'], size=n, p=[0.8,0.1,0.1]),
    'ts': timestamps
})
tx.to_csv('data/transactions.csv', index=False)
print('Wrote data/users.csv and data/transactions.csv')

In [ ]:
# Cell 4: Read CSV with Spark and inspect
df_users = spark.read.option('header',True).option('inferSchema',True).csv('data/users.csv')
df_tx = spark.read.option('header',True).option('inferSchema',True).csv('data/transactions.csv')

print('Users schema:')
df_users.printSchema()
df_users.show(5,truncate=False)

print('Transactions schema:')
df_tx.printSchema()
df_tx.show(5,truncate=False)

In [ ]:
# Cell 5: Basic transformations - select, filter, withColumn
from pyspark.sql.functions import col, to_timestamp, expr

# Convert ts column to proper timestamp type
df_tx2 = df_tx.withColumn('event_time', to_timestamp(col('ts'))).drop('ts')

# Keep only transactions > 50 and normalize amount to EUR (simple rates)
df_tx_small = (df_tx2
               .select('tx_id','user_id','amount','currency','event_time')
               .filter(col('amount') > 50.0)
               .withColumn('amount_eur', expr("CASE WHEN currency='GBP' THEN amount*1.15 WHEN currency='USD' THEN amount*0.95 ELSE amount END"))
              )

df_tx_small.show(5,truncate=False)

In [ ]:
# Cell 6: Join with users and inspect
# Enrich transactions with user information

df_join = df_tx_small.join(df_users, on='user_id', how='left')
df_join.select('tx_id','user_id','name','country','amount','amount_eur').show(8,truncate=False)

In [ ]:
# Cell 7: Aggregations - total and average per user
from pyspark.sql.functions import sum as _sum, avg as _avg, count as _count

agg_user = (df_join.groupBy('user_id','name')
            .agg(_count('tx_id').alias('n_tx'),
                 _sum('amount_eur').alias('total_eur'),
                 _avg('amount_eur').alias('avg_eur'))
            .orderBy(col('total_eur').desc())
           )
agg_user.show(truncate=False)

In [ ]:
# Cell 8: Explain plan and caching
print('Execution plan (detailed):')
agg_user.explain(True)

# Cache result if you will reuse it
agg_user_cached = agg_user.cache()
print('Trigger cache (count):', agg_user_cached.count())

In [ ]:
# Cell 9: Simple UDF example (categorize users)
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def tier(total):
    if total >= 2000:
        return 'gold'
    elif total >= 500:
        return 'silver'
    else:
        return 'bronze'

tier_udf = udf(tier, StringType())
agg_with_tier = agg_user_cached.withColumn('tier', tier_udf(col('total_eur')))
agg_with_tier.show(truncate=False)

In [ ]:
# Cell 10: Write results as Parquet partitioned by tier
out_path = 'output/parquet/agg_user'
if os.path.exists(out_path):
    shutil.rmtree(out_path)

(agg_with_tier.repartition(1)
 .write
 .mode('overwrite')
 .partitionBy('tier')
 .parquet(out_path)
)
print('Wrote parquet to', out_path)

In [ ]:
# Cell 11: Read parquet and export a CSV summary for submission
df_parquet = spark.read.parquet(out_path)
pdf = df_parquet.toPandas().sort_values('total_eur', ascending=False)
pdf.to_csv('output/results.csv', index=False)
print('Exported output/results.csv')

In [ ]:
# Cell 12: Simple plot of totals per user (small result)
import matplotlib.pyplot as plt
plt.figure(figsize=(8,4))
plt.bar(pdf['name'], pdf['total_eur'])
plt.title('Total EUR per user')
plt.ylabel('Total EUR')
plt.xlabel('User')
plt.tight_layout()
plt.savefig('output/total_per_user.png', dpi=150)
plt.show()
print('Saved output/total_per_user.png')

## Exercises (to implement)
1. Change the exchange rates (GBP=1.2 and USD=1.0) and save the top 3 users (by `total_eur`) as `output/top3_users.csv`
2. Compute total transaction amount per country and identify the top country.
3. Show the top-3 users separately for USD and GBP transactions. How does it compare to the top-3 in EUR
4. Increase USD and GBP rates by 10%. How does the ranking change?

In [ ]:
# Cell 13: Clean shutdown
spark.stop()
print('Spark stopped')